In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')
import openpyxl
from openpyxl.styles import Border, Side, Font, Alignment, PatternFill


In [22]:
# 设置随机种子
np.random.seed(42)

In [29]:
# 数据路径
data_path = r"C:\Users\sun\Desktop\在写\回归结果\data.csv"

In [30]:
# 输出结果路径
output_dir = os.path.dirname(data_path)
output_path = os.path.join(output_dir, "DDML_IV_基准回归结果.xlsx")

In [31]:
# 读取数据
print("正在读取数据...")
try:
    data = pd.read_csv(data_path)
    print(f"数据读取成功，共 {data.shape[0]} 行，{data.shape[1]} 列")
except Exception as e:
    print(f"数据读取失败: {e}")
    raise

正在读取数据...
数据读取成功，共 3069 行，25 列


In [32]:
# 打印列名，帮助调试
print("数据集中的列名:")
print(data.columns.tolist())

数据集中的列名:
['id', 'control1', 'control2', 'control3', 'pro', 'year', 'year_2014', 'RLDS', 'DIF', 'DT', 'DID', 'UCI', 'ULGUE', 'GOV', 'INF', 'Popd', 'Trade', 'GPatent', 'ERS', 'GOV2', 'INF2', 'Popd2', 'Trade2', 'GPatent2', 'ERS2']


In [33]:
# 检查RLDS变量是否存在
if 'RLDS' not in data.columns:
    print("警告: 数据中没有'RLDS'变量，工具变量法无法实施")
    raise ValueError("缺少RLDS变量，请检查数据")


In [34]:
# 数据预处理
print("正在进行数据预处理...")
# 检查并处理缺失值
missing_values = data.isnull().sum()
if missing_values.sum() > 0:
    print(f"数据存在缺失值，总计: {missing_values.sum()} 个")
    print(missing_values[missing_values > 0])
    # 使用中位数填充数值型缺失值
    for col in data.select_dtypes(include=[np.number]).columns:
        if data[col].isnull().sum() > 0:
            data[col] = data[col].fillna(data[col].median())
    print("已使用中位数填充数值型缺失值")

正在进行数据预处理...


In [35]:
# 生成工具变量Z，与STATA代码保持一致
print("正在生成工具变量...")
data['Z'] = data['RLDS'] * (data['year'] - 2010)
print(f"已生成工具变量 Z = RLDS * (year - 2010)")

正在生成工具变量...
已生成工具变量 Z = RLDS * (year - 2010)


In [ ]:
# 定义变量
Y = 'ULGUE'  # 因变量
D = 'DIF'    # 核心解释变量(内生变量)
Z = 'Z'      # 工具变量
X = ['GOV', 'INF', 'Popd', 'Trade', 'GPatent', 'ERS']  # 控制变量

In [ ]:
# 检查变量是否都在数据集中
all_vars = [Y, D, Z] + X
missing_vars = [var for var in all_vars if var not in data.columns]
if missing_vars:
    print(f"以下变量在数据集中不存在: {missing_vars}")
    print("请检查变量名或数据集结构，以下是可用的列名:")
    print(data.columns.tolist())
    raise ValueError("变量不存在，请检查变量名")

In [ ]:
# 检查数据中是否有年份和城市标识符
if 'year' not in data.columns:
    print("警告: 数据中没有'year'列，无法添加年份固定效应")
    has_year = False
else:
    has_year = True
    print(f"发现年份变量，共有 {data['year'].nunique()} 个不同年份")

if 'id' not in data.columns:
    print("警告: 数据中没有'id'列，无法添加城市固定效应")
    has_id = False
else:
    has_id = True
    print(f"发现城市ID变量，共有 {data['id'].nunique()} 个不同城市")

In [ ]:
# 创建固定效应变量
if has_year:
    # 创建年份哑变量
    year_dummies = pd.get_dummies(data['year'], prefix='year', drop_first=True)
    data = pd.concat([data, year_dummies], axis=1)
    year_dummy_cols = year_dummies.columns.tolist()
    X = X + year_dummy_cols
    print(f"已添加 {len(year_dummy_cols)} 个年份哑变量")

In [ ]:
if has_id:
    # 创建城市哑变量
    id_dummies = pd.get_dummies(data['id'], prefix='id', drop_first=True)
    data = pd.concat([data, id_dummies], axis=1)
    id_dummy_cols = id_dummies.columns.tolist()
    X = X + id_dummy_cols
    print(f"已添加 {len(id_dummy_cols)} 个城市哑变量")

print(f"最终使用的控制变量数量: {len(X)}")


In [ ]:
# 实现双重去偏机器学习工具变量法(DDML-IV)
print("开始实施双重去偏机器学习工具变量法(DDML-IV)...")
